In [ ]:
# Modelagem Baseline — Predição de Alfabetização

Este notebook roda do zero, sem depender de estado de outros notebooks:
carrega a base, aplica a engenharia de atributos (já incluindo o
enriquecimento externo — Censo Escolar e ADH), separa treino/teste por
município, e treina dois modelos de comparação — Regressão Logística
(baseline simples) e Random Forest (não-linear) — para saber se a
complexidade do algoritmo ou a riqueza das features é o fator limitante.

In [ ]:
## Carregamento e construção do dataset

`build_dataset()` aplica tudo que já validamos: filtro de escopo (só
alunos avaliados, exclui rede Privada), flags de ausência, e o merge com
Censo Escolar (por ano+município) e ADH (por município, proxy de 2010).

In [1]:
import sys
sys.path.append("..")

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

from src.preprocessing.build_features import build_dataset
from src.preprocessing.split import split_by_municipio
from src.preprocessing.pipeline import NUMERIC_WITH_MISSING, NUMERIC_COMPLETE, CATEGORICAL
from src.modeling.pipeline import build_model_pipeline

df = pd.read_parquet("../data/raw/features_alunos_ml.parquet")
dataset = build_dataset(df)
print("Linhas no dataset:", len(dataset))
print("Colunas:", list(dataset.columns))

Linhas no dataset: 3355822
Colunas: ['ano', 'sigla_uf', 'rede_label', 'taxa_alfabetizacao_municipio', 'meta_alfabetizacao_municipio_ano', 'pct_biblioteca', 'pct_sala_leitura', 'pct_internet_aprendizagem', 'pct_laboratorio_informatica', 'pct_agua_potavel', 'pct_esgoto_rede_publica', 'pct_energia_rede_publica', 'pct_alimentacao', 'pct_parque_infantil', 'media_alunos_por_turma', 'pct_com_pedagogo', 'pct_com_psicologo', 'idhm', 'idhm_e', 'renda_pc', 'prop_pobreza_criancas', 'taxa_analfabetismo_15_mais', 'taxa_criancas_fora_escola_6_14', 'taxa_municipio_ausente', 'meta_ausente', 'censo_escolar_ausente', 'adh_ausente', 'label_alfabetizado', 'id_municipio', 'peso_aluno']


In [ ]:
## Split treino/teste por município

Sempre antes de qualquer ajuste de imputador/encoder — evita que
estatísticas do teste vazem pro treino.

In [2]:
FEATURE_COLS = NUMERIC_WITH_MISSING + NUMERIC_COMPLETE + CATEGORICAL

train, test = split_by_municipio(dataset)
print("Treino:", train.shape, "| Teste:", test.shape)
print("Balanceamento treino:", train["label_alfabetizado"].mean().round(4))
print("Balanceamento teste:", test["label_alfabetizado"].mean().round(4))

Treino: (2682972, 30) | Teste: (672850, 30)
Balanceamento treino: 0.591
Balanceamento teste: 0.5928


In [ ]:
## Modelo 1 — Regressão Logística (baseline)

Peso amostral (`peso_aluno`) passado via `classifier__sample_weight`
porque o modelo está dentro de um `Pipeline` — o prefixo `classifier__`
direciona o parâmetro pra etapa certa.

In [3]:
baseline = build_model_pipeline(LogisticRegression(max_iter=1000, random_state=42))
baseline.fit(
    train[FEATURE_COLS],
    train["label_alfabetizado"],
    classifier__sample_weight=train["peso_aluno"],
)
print("Treinado.")

Treinado.


In [4]:
y_pred_baseline = baseline.predict(test[FEATURE_COLS])
print(classification_report(test["label_alfabetizado"], y_pred_baseline, target_names=["Não alfabetizado", "Alfabetizado"]))
print("Matriz de confusão:")
print(confusion_matrix(test["label_alfabetizado"], y_pred_baseline))

                  precision    recall  f1-score   support

Não alfabetizado       0.55      0.46      0.50    273955
    Alfabetizado       0.67      0.74      0.70    398895

        accuracy                           0.62    672850
       macro avg       0.61      0.60      0.60    672850
    weighted avg       0.62      0.62      0.62    672850

Matriz de confusão:
[[127231 146724]
 [105651 293244]]


In [ ]:
## Modelo 2 — Random Forest

Mesmo conjunto de features enriquecido, agora com um modelo capaz de
capturar interações não-lineares entre variáveis (ex.: o efeito de
`idhm` pode depender da UF, algo que a Regressão Logística não representa
sem termos de interação explícitos).

In [6]:
rf = build_model_pipeline(
    RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1)
)
rf.fit(
    train[FEATURE_COLS],
    train["label_alfabetizado"],
    classifier__sample_weight=train["peso_aluno"],
)
print("Treinado.")

Treinado.


In [7]:
y_pred_rf = rf.predict(test[FEATURE_COLS])
print(classification_report(test["label_alfabetizado"], y_pred_rf, target_names=["Não alfabetizado", "Alfabetizado"]))
print("Matriz de confusão:")
print(confusion_matrix(test["label_alfabetizado"], y_pred_rf))

                  precision    recall  f1-score   support

Não alfabetizado       0.55      0.42      0.48    273955
    Alfabetizado       0.66      0.77      0.71    398895

        accuracy                           0.63    672850
       macro avg       0.61      0.59      0.59    672850
    weighted avg       0.62      0.63      0.61    672850

Matriz de confusão:
[[114306 159649]
 [ 91925 306970]]


In [ ]:
## Conclusão: o teto é estrutural, não de dados nem de algoritmo

Testamos duas hipóteses pro baseline fraco: (1) faltava um modelo melhor —
descartada (Random Forest não supera Regressão Logística); (2) faltavam
mais variáveis — também descartada (18 features novas de Censo Escolar e
ADH não mudaram a acurácia).

A explicação mais provável: quase todas as nossas features são
**agregadas por município** — o mesmo valor de `idhm`, `pct_biblioteca`,
`taxa_alfabetizacao_municipio` se repete para todos os alunos daquele
município naquele ano. Uma variável constante dentro de um grupo só
consegue explicar a diferença **entre** municípios, nunca a diferença
**entre alunos do mesmo município** — e é justamente aí que mora a maior
parte da variação real de quem está alfabetizado ou não (fatores
individuais, familiares, de sala de aula, que esta base não captura).

Isso não invalida o enriquecimento — ele enriquece a *explicação* do
padrão territorial (é isso que vamos explorar na interpretabilidade), só
não move o teto da *predição individual* pra além de ~63%.